# Getting Started with Agentic Assistants

This notebook will help you get started with the Agentic Assistants framework. We'll cover:

1. **Environment Setup** - Verifying your installation
2. **Configuration** - Understanding the config system
3. **Ollama Basics** - Managing models and chat
4. **MLFlow Tracking** - Basic experiment tracking
5. **OpenTelemetry** - Observability basics

## Prerequisites

Before running this notebook, ensure you have:
- Python 3.10+ installed
- Poetry installed (`pip install poetry`)
- Ollama installed (https://ollama.ai)
- Project dependencies installed (`poetry install`)


## 1. Environment Setup

Let's verify that everything is installed correctly.


In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ is required"

# Check core imports
try:
    from agentic_assistants import __version__, AgenticConfig, OllamaManager
    print(f"✓ Agentic Assistants v{__version__} installed")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("Run: poetry install")

# Check optional dependencies
deps = {
    "mlflow": "MLFlow",
    "opentelemetry": "OpenTelemetry", 
    "crewai": "CrewAI",
    "langgraph": "LangGraph",
}

for module, name in deps.items():
    try:
        __import__(module)
        print(f"✓ {name} installed")
    except ImportError:
        print(f"✗ {name} not installed")


## 2. Configuration

The framework uses a hierarchical configuration system with Pydantic Settings. Configuration can come from:
1. Environment variables
2. `.env` file
3. Programmatic overrides


In [ ]:
from agentic_assistants import AgenticConfig

# Load configuration (reads from environment and .env)
config = AgenticConfig()

# View all settings
print("Current Configuration:")
print("-" * 40)
for key, value in config.to_dict().items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")


In [ ]:
# You can override settings programmatically
custom_config = AgenticConfig(
    mlflow_enabled=False,  # Disable MLFlow tracking
    log_level="DEBUG",
)

print(f"MLFlow enabled: {custom_config.mlflow_enabled}")
print(f"Log level: {custom_config.log_level}")


## 3. Ollama Management

The `OllamaManager` class provides methods to start/stop Ollama and manage models.


In [ ]:
from agentic_assistants import OllamaManager

# Initialize the manager
ollama = OllamaManager(config)

# Check status
status = ollama.get_status()
print("Ollama Status:")
print(f"  Running: {status['running']}")
print(f"  Host: {status['host']}")

if not status['running']:
    print("\n⚠️  Ollama is not running!")
    print("Start it with: ollama.start() or run 'ollama serve' in terminal")


In [ ]:
# List available models (only works if Ollama is running)
if ollama.is_running():
    models = ollama.list_models()
    print(f"Available models ({len(models)}):")
    for model in models:
        print(f"  - {model.name} ({model.size_gb:.1f} GB)")
    
    if not models:
        print("\nNo models found. Pull one with:")
        print("  ollama.pull_model('llama3.2')")
else:
    print("Start Ollama first to list models")


In [ ]:
# Simple chat example (only works if Ollama is running with a model)
if ollama.is_running() and ollama.list_models():
    messages = [
        {"role": "user", "content": "Hello! What can you help me with today?"}
    ]
    
    print("Sending chat request...")
    response = ollama.chat(messages)
    
    print("\nResponse:")
    print(response["message"]["content"])
else:
    print("Ensure Ollama is running with a model to test chat")


## 4. MLFlow Tracking

MLFlow integration is built-in for experiment tracking. It can be enabled/disabled at runtime.


In [ ]:
from agentic_assistants.core import MLFlowTracker

# Initialize tracker (respects config.mlflow_enabled)
tracker = MLFlowTracker(config)

print(f"MLFlow enabled: {tracker.enabled}")
print(f"Tracking URI: {config.mlflow.tracking_uri}")
print(f"Experiment: {config.mlflow.experiment_name}")


In [ ]:
# Example: Track an experiment
# Note: MLFlow server should be running for this to work
# Start it with: agentic mlflow start

import time

with tracker.start_run(run_name="notebook-example") as run:
    # Log parameters
    tracker.log_param("model", config.ollama.default_model)
    tracker.log_param("example_type", "notebook")
    
    # Simulate some work
    start = time.time()
    time.sleep(0.5)  # Simulated processing
    duration = time.time() - start
    
    # Log metrics
    tracker.log_metric("duration_seconds", duration)
    tracker.log_metric("success", 1)
    
    # Log text artifact
    tracker.log_text("This is a test output from the notebook.", "output/test.txt")
    
    if run:
        print(f"Run ID: {run.info.run_id}")
        print(f"View at: {tracker.get_run_url()}")
    else:
        print("MLFlow tracking disabled or server not running")


## 5. OpenTelemetry Basics

OpenTelemetry provides distributed tracing and metrics collection.


In [ ]:
from agentic_assistants.core import TelemetryManager, get_tracer

# Initialize telemetry
telemetry = TelemetryManager(config)

print(f"Telemetry enabled: {telemetry.enabled}")
print(f"Service name: {telemetry.service_name}")
print(f"OTLP endpoint: {config.telemetry.exporter_otlp_endpoint}")


In [ ]:
# Example: Create a span for tracing
# Note: OTEL collector should be running for traces to be exported
# Start with: docker-compose up -d

with telemetry.span("notebook-operation", attributes={"step": "demo"}) as span:
    # Simulate work
    time.sleep(0.1)
    span.set_attribute("result", "success")
    print("Span created (traces exported if collector is running)")


## Next Steps

Now that you've verified your setup, explore these notebooks:

1. **02_crewai_basics.ipynb** - Build multi-agent teams with CrewAI
2. **03_langgraph_basics.ipynb** - Create stateful workflows with LangGraph
3. **04_mlflow_experiments.ipynb** - Advanced experiment tracking and comparison

Or run the example scripts:
```bash
agentic run examples/simple_ollama_chat.py
agentic run examples/crewai_research_team.py
agentic run examples/langgraph_workflow.py
```
